# Guard versus task-fine-tuned SmolLM2

Compare SystemOne Guard (756,248 parameters) with SmolLM2-135M-Instruct adapted using LoRA (134,515,008 merged parameters). The updated guard uses 90,000 training and 3,500 validation examples; frozen SmolLM2 LoRA retains its original 10,000/1,500 splits. The data budgets differ. Test, challenge, manual probes and the frozen new wording set are excluded from gradient updates and epoch selection. The inspected manual errors did motivate the redesign.

Use a Python environment with the project's `dev`, `cpu`, and `notebook` extras for saved results. GPU reproduction uses `make setup-gpu` and its separate `.venv-gpu` environment. The cells below leave training and evaluation disabled until explicitly enabled. See `docs/EVALUATION.md` for exact timing boundaries and effect formulas.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
print(ROOT)

## Optional GPU reproduction

Reuse the completed adapter in `artifacts/smollm2_lora/best`, or train it once with `make finetune`. Evaluation requires that adapter. Both models are resident on the same GPU; timing order alternates over matched examples after warmup. Quality uses all examples, while batch-one timing uses 100/100/24/100 matched requests. Batched quality is never presented as request latency.

In [ ]:
import subprocess

RUN_FINETUNING = False
RUN_EVALUATION = False

if RUN_FINETUNING or RUN_EVALUATION:
    subprocess.run(["make", "setup-gpu"], cwd=ROOT, check=True)

if RUN_FINETUNING:
    subprocess.run(["make", "finetune"], cwd=ROOT, check=True)

if RUN_EVALUATION:
    for split in ("test", "challenge", "manual_probes", "generalization"):
        subprocess.run(
            ["make", "benchmark", f"ARGS=--split {split}"], cwd=ROOT, check=True
        )

## Validate and read the complete measurements

The loader recomputes each summary and effect from raw paired records. It rejects stale summaries, extra methods, differing IDs or gold labels, and mismatched timing subsets.

In [ ]:
import pandas as pd
from IPython.display import display

from systemone_guard.benchmarking.reporting import load_reports

reports = load_reports(ROOT / "experiments")
summary_rows = [
    {"split": split, **summary, "latency_n": summary.get("latency_n", summary["n"])}
    for split, report in reports.items()
    for summary in report["summaries"]
    if summary is not None
]
display(
    pd.DataFrame(summary_rows)[
        [
            "split",
            "method",
            "n",
            "accuracy",
            "macro_f1",
            "nll",
            "brier",
            "ece_10",
            "aux_accuracy",
            "invalid_rate",
            "latency_n",
            "median_latency_ms",
            "p95_latency_ms",
            "mean_latency_ms",
        ]
    ]
)

## Accuracy changes and latency savings

Accuracy gain is **LoRA minus Guard**, in percentage points; relative gain has a different denominator. Guard latency reduction uses LoRA latency as the baseline. Speedup is LoRA latency divided by Guard latency. Negative differences are retained, and all effects use exact counts or raw timings before rounding.

In [ ]:
accuracy_rows = [
    {
        "split": split,
        **{
            key: report["comparison"][key]
            for key in (
                "quality_n",
                "guard_correct",
                "smollm_correct",
                "additional_correct",
                "accuracy_gain_pp",
                "accuracy_relative_gain_percent",
                "accuracy_gain_ci95_pp",
            )
        },
    }
    for split, report in reports.items()
]
latency_rows = [
    {
        "split": split,
        "statistic": statistic,
        "latency_n": report["comparison"]["latency_n"],
        **effect,
    }
    for split, report in reports.items()
    for statistic, effect in report["comparison"]["latency"].items()
]
display(pd.DataFrame(accuracy_rows))
display(pd.DataFrame(latency_rows))

## Free-generation diagnostic and incorrect examples

Constrained decoding supplies the main comparison. Greedy free generation is a separate four-word format check, with invalid output counted as an error. Its latency does not determine the main speedup.

In [ ]:
generation_rows = [
    {"split": row["split"], "method": row["method"], **row["generation"]}
    for row in summary_rows
    if "generation" in row
]
errors = [
    {
        "split": split,
        "id": row["id"],
        "method": row["method"],
        "gold": row["gold"],
        "prediction": row["prediction"],
        "probabilities": row["probabilities"],
    }
    for split, report in reports.items()
    for row in report["records"]
    if row["prediction"] != row["gold"]
]
display(pd.DataFrame(generation_rows))
display(pd.DataFrame(errors))

## Regenerate every comparison figure

The six canonical figures include both models: accuracy, latency, calibration, confusion, auxiliary heads, and recorded training traces. Guard and LoRA use different training objectives; training losses are shown separately.

In [ ]:
from IPython.display import Image
from scripts.plots import create_plots

create_plots(ROOT / "experiments", ROOT / "figures")

for name in (
    "comparison",
    "latency",
    "calibration",
    "confusion",
    "auxiliary_accuracy",
    "training",
):
    display(Image(filename=str(ROOT / "figures" / f"{name}.png")))

## Interpretation

Read the exact counts and paired intervals before claiming an accuracy improvement. A small synthetic difference and a gain on 24 manually selected probes have different implications. The original guard missed all ten unfamiliar attacks. Those inspected probes are now a development benchmark; the new 256-example final wording set was frozen before candidate training. Both models still have failures.

This compares task adaptation and inference cost for two architectures with different parameter counts and prior training. It does not establish production security performance or match total training compute. Timing changes with hardware, load, and runtime implementation.
Guard reaches 24/24 development probes and 232/256 on final wording. It misses 24/72 untrusted attacks that LoRA detects. The 95% overall wording target remains unmet. Training workers now have a 2 GiB RAM cap and stream disk-cached features.
